# Main Results: Layerwise Plots

This notebook loads the Excel result workbooks from `results/*.xlsx` and renders Experiment 2 layerwise depth-profile plot variants for every probe, family-mean depth plots, LTX noise/block heatmaps, and an all-probes appendix grid.


In [ ]:
from pathlib import Path
import os
import re
import sys

os.environ.setdefault('MPLCONFIGDIR', '/tmp/probe4physics_matplotlib')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp/probe4physics_cache')
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
Path(os.environ['XDG_CACHE_HOME']).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'This notebook requires matplotlib. Use the probe4physics-gpu environment or install matplotlib.'
    ) from exc
try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    class Markdown(str):
        pass
    def display(obj):
        print(obj)

try:
    import seaborn as sns
except Exception:
    sns = None

pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 180)
if sns is not None:
    sns.set_theme(style='whitegrid', context='notebook')
else:
    plt.rcParams.update({'axes.grid': True, 'grid.alpha': 0.25})

def find_repo_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    for candidate in [current, *current.parents]:
        if (candidate / 'results').exists() and (candidate / 'run.py').exists():
            return candidate
    raise FileNotFoundError('Could not find repository root containing results/ and run.py')

REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / 'notebooks'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
import importlib
import plot_colors
importlib.reload(plot_colors)
from plot_colors import BACKBONE_COLORS, FAMILY_COLORS, MODEL_COLORS, PROBE_COLORS, PROBE_LABEL_COLORS, REFERENCE_COLORS
from seed_variance_loader import load_ready_seed_summary, report_seed_coverage

RESULTS_DIR = REPO_ROOT / 'results'
MVP_XLSX = RESULTS_DIR / 'mvp_primary_db.xlsx'
INTPHYS_XLSX = RESULTS_DIR / 'intphys_primary_db.xlsx'
VIS_XLSX = RESULTS_DIR / 'visualization_ready_results.xlsx'

HEADLINE_PROBE = 'mlp'
PROBE_ORDER = ['linear', 'mlp', 'temporal_attn']
PROBE_LABEL = {'linear': 'Linear', 'mlp': 'MLP', 'temporal_attn': 'Attentive'}
DATASET_ORDER = ['mvp', 'intphys2']
PANEL_DATASET_ORDER = ['mvp', 'intphys2']
DATASET_LABEL = {'mvp': 'MVP', 'intphys2': 'IntPhys2'}
PRIMARY_LABEL = {'mvp': 'pair consistency', 'intphys2': 'VOE accuracy'}
RANDOM_BASELINE = {'mvp': 25.0, 'intphys2': 100.0 / 6.0}

MODEL_ORDER = ['jepa_v1', 'jepa_v2', 'jepa_v2_1', 'videomae', 'videomae_v2', 'ltx_video']
MODEL_ORDER_INDEX = {key: idx for idx, key in enumerate(MODEL_ORDER)}
MODEL_META = {
    'jepa_v1': {'label': 'V-JEPA', 'family': 'V-JEPA', 'objective': 'self-supervised predictive video representation', 'paper_backbone': 'ViT-H/16'},
    'jepa_v2': {'label': 'V-JEPA 2', 'family': 'V-JEPA', 'objective': 'self-supervised predictive video representation', 'paper_backbone': 'ViT-G/16'},
    'jepa_v2_1': {'label': 'V-JEPA 2.1', 'family': 'V-JEPA', 'objective': 'self-supervised predictive video representation', 'paper_backbone': 'ViT-Gigantic/16'},
    'videomae': {'label': 'VideoMAE', 'family': 'VideoMAE', 'objective': 'masked video autoencoding', 'paper_backbone': 'ViT-H/16'},
    'videomae_v2': {'label': 'VideoMAE-v2', 'family': 'VideoMAE', 'objective': 'masked video autoencoding', 'paper_backbone': 'ViT-G/16'},
    'ltx_video': {'label': 'LTX-Video', 'family': 'Diffusion video model', 'objective': 'video diffusion generation', 'paper_backbone': 'LTX-13B'},
}
SHEET_TO_MODEL_KEY = {
    'V-JEPA': 'jepa_v1',
    'V-JEPA 2': 'jepa_v2',
    'V-JEPA 2.1': 'jepa_v2_1',
    'VideoMAE': 'videomae',
    'VideoMAE-v2': 'videomae_v2',
    'LTX-Video': 'ltx_video',
}

for path in [MVP_XLSX, INTPHYS_XLSX]:
    if not path.exists():
        raise FileNotFoundError(f'Missing required workbook: {path}')


SEED_MERGE_KEYS = ['dataset', 'experiment', 'model_label', 'backbone', 'probe', 'layer_key']
SEED_STAT_COLUMNS = [
    'n_seeds', 'seeds', 'seed_source',
    'test_primary_mean', 'test_primary_std', 'test_accuracy_mean', 'test_accuracy_std',
    'val_primary_mean', 'val_primary_std', 'val_accuracy_mean', 'val_accuracy_std',
]

def dataset_key(value):
    text = str(value).strip().lower()
    if text == 'intphys2':
        return 'intphys2'
    if text == 'mvp':
        return 'mvp'
    return text

def normalized_layer_label(value):
    if value is None or pd.isna(value):
        return ''
    text = re.sub(r'\s+', ' ', str(value).strip().lower())
    return text

def numeric_layer_key(value):
    number = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
    if pd.isna(number):
        return ''
    return f'id:{number:g}'

def label_layer_key(value):
    label = normalized_layer_label(value)
    return f'label:{label}' if label else ''

def row_layer_key(row):
    return numeric_layer_key(row.get('layer')) or label_layer_key(row.get('layer_label'))

def load_seed_summary():
    return load_ready_seed_summary(
        RESULTS_DIR,
        seed_merge_keys=SEED_MERGE_KEYS,
        seed_stat_columns=SEED_STAT_COLUMNS,
    )

def attach_seed_stats(df):
    seeds = load_seed_summary()
    base = df.drop(columns=[c for c in [*SEED_STAT_COLUMNS, 'layer_key'] if c in df.columns], errors='ignore').copy()
    base['layer'] = pd.to_numeric(base['layer'], errors='coerce')
    if {'model_label', 'backbone'}.issubset(base.columns):
        ltx_mask = base['backbone'].isin(['LTX-2B', 'LTX-13B'])
        base.loc[ltx_mask, 'model_label'] = base.loc[ltx_mask, 'backbone']
    base['layer_key'] = base.apply(row_layer_key, axis=1)
    merged = base.merge(seeds, on=SEED_MERGE_KEYS, how='left', validate='many_to_one')
    merged['plot_test_primary'] = merged['test_primary_mean'].combine_first(merged['test_primary'])
    merged['plot_test_primary_std'] = pd.to_numeric(merged['test_primary_std'], errors='coerce')
    merged['plot_test_accuracy'] = merged['test_accuracy_mean'].combine_first(merged['test_accuracy'])
    merged['plot_test_accuracy_std'] = pd.to_numeric(merged['test_accuracy_std'], errors='coerce')
    merged['plot_val_primary'] = merged['val_primary_mean'].combine_first(merged['val_primary'])
    merged['plot_val_primary_std'] = pd.to_numeric(merged['val_primary_std'], errors='coerce')
    return merged

def yerr_or_none(values):
    err = pd.to_numeric(values, errors='coerce')
    return None if err.isna().all() else err.to_numpy(dtype=float)


def canonical_probe(value):
    text = str(value).strip().lower()
    if text in {'linear', 'lin'}:
        return 'linear'
    if text == 'mlp':
        return 'mlp'
    if text in {'attentive', 'temporal_attn', 'temporal attentive', 'temporal-attn'}:
        return 'temporal_attn'
    return text or None

def parse_ltx_slot(label):
    if label is None or pd.isna(label):
        return (np.nan, np.nan)
    match = re.search(r'noise_([0-9.]+)_block_(\d+)', str(label))
    if not match:
        return (np.nan, np.nan)
    return (float(match.group(1)), int(match.group(2)))

def relative_depth_from_label(label, fallback_rank=None, fallback_count=None):
    text = str(label).strip().lower()
    if text in {'final layer', 'last', 'final'}:
        return 1.0
    match = re.search(r'0\.(25|5|50|75)', text)
    if match:
        raw = match.group(0)
        return float(raw)
    if fallback_rank is not None and fallback_count not in (None, 0):
        return float(fallback_rank) / float(fallback_count)
    return np.nan

def parse_primary_workbook(path, dataset):
    frames = []
    xl = pd.ExcelFile(path)
    for sheet in xl.sheet_names:
        raw = xl.parse(sheet)
        cols = list(raw.columns)
        starts = [
            idx for idx, col in enumerate(cols)
            if not str(col).startswith('Unnamed') and 'primary metric' in str(col).lower()
        ]
        for start in starts:
            end = min(start + 11, len(cols))
            title = str(cols[start])
            backbone = title.split(' - primary metric:', 1)[0].strip()
            block = raw.iloc[:, start:end].copy()
            headers = [str(item).strip() for item in block.iloc[0].tolist()]
            block = block.iloc[1:].copy()
            block.columns = headers
            block = block.dropna(how='all')
            if 'Experiment' not in block.columns:
                continue
            block = block[block['Experiment'].notna()].copy()
            if block.empty:
                continue
            block['dataset'] = dataset
            block['workbook'] = path.name
            block['sheet'] = sheet
            block['model_key'] = SHEET_TO_MODEL_KEY.get(sheet, sheet)
            block['backbone'] = backbone
            frames.append(block)
    if not frames:
        raise ValueError(f'No parseable result blocks found in {path}')
    return pd.concat(frames, ignore_index=True, sort=False)

def load_excel_results():
    raw = pd.concat([
        parse_primary_workbook(MVP_XLSX, 'mvp'),
        parse_primary_workbook(INTPHYS_XLSX, 'intphys2'),
    ], ignore_index=True, sort=False)
    rename = {
        'Experiment': 'experiment',
        'Probe': 'probe_raw',
        'Layer': 'layer',
        'Layer Label': 'layer_label',
        'Selected LR': 'selected_lr',
        'Train Accuracy': 'train_accuracy',
        'Val Accuracy': 'val_accuracy',
        'Test Accuracy': 'test_accuracy',
        'Train Primary Metric': 'train_primary',
        'Val Primary Metric': 'val_primary',
        'Test Primary Metric': 'test_primary',
    }
    df = raw.rename(columns=rename).copy()
    df['probe'] = df['probe_raw'].map(canonical_probe)
    df['model_label'] = df['model_key'].map(lambda key: MODEL_META[key]['label'])
    df['family'] = df['model_key'].map(lambda key: MODEL_META[key]['family'])
    df['pretraining_objective'] = df['model_key'].map(lambda key: MODEL_META[key]['objective'])
    df['model_order'] = df['model_key'].map(MODEL_ORDER_INDEX).fillna(999).astype(int)
    for col in ['layer', 'selected_lr', 'train_accuracy', 'val_accuracy', 'test_accuracy', 'train_primary', 'val_primary', 'test_primary']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['layer_label'] = df['layer_label'].astype('string')
    slots = df['layer_label'].map(parse_ltx_slot)
    df['ltx_noise_level'] = [slot[0] for slot in slots]
    df['ltx_block'] = [slot[1] for slot in slots]
    df['is_ltx'] = df['model_key'].eq('ltx_video')
    df['paper_backbone'] = df['model_key'].map(lambda key: MODEL_META[key]['paper_backbone'])
    df['is_paper_backbone'] = df['backbone'].eq(df['paper_backbone'])
    df = add_relative_depth(df)
    df = attach_seed_stats(df)
    return df

def add_relative_depth(df):
    df = df.copy()
    df['relative_depth'] = np.nan
    for keys, group in df[~df['is_ltx']].groupby(['dataset', 'model_key', 'backbone', 'probe'], dropna=False):
        group = group.sort_values(['layer', 'layer_label'])
        count = len(group)
        for rank, idx in enumerate(group.index, start=1):
            df.loc[idx, 'relative_depth'] = relative_depth_from_label(df.loc[idx, 'layer_label'], rank, count)
    df.loc[df['is_ltx'], 'relative_depth'] = np.nan
    df['relative_depth_label'] = df['relative_depth'].map(lambda value: pd.NA if pd.isna(value) else f'{value:.2f}')
    return df

def paper_results(df):
    return df[df['is_paper_backbone']].copy()

def select_best_layers(df):
    candidates = df.dropna(subset=['plot_test_primary']).copy()
    ordered = candidates.sort_values([
        'dataset', 'probe', 'model_key', 'backbone',
        'plot_test_primary', 'plot_test_accuracy', 'test_primary', 'val_primary',
    ])
    best = ordered.groupby(['dataset', 'probe', 'model_key', 'backbone'], group_keys=False).tail(1).copy()
    best['primary_score'] = best['plot_test_primary']
    best['primary_score_std'] = best['plot_test_primary_std']
    best['best_validation_score'] = best['plot_val_primary']
    best['test_accuracy_score'] = best['plot_test_accuracy']
    best['test_accuracy_score_std'] = best['plot_test_accuracy_std']
    return best.sort_values(['dataset', 'probe', 'model_order']).reset_index(drop=True)

def show_table(df, title):
    display(Markdown(f'**{title}**'))
    display(df.style.format(precision=2, na_rep=''))

def row_for(df, dataset, model_key, probe=None):
    mask = df['dataset'].eq(dataset) & df['model_key'].eq(model_key)
    if probe is not None:
        mask &= df['probe'].eq(probe)
    rows = df[mask]
    if rows.empty:
        return None
    return rows.iloc[0]

def value_or_nan(row, column):
    if row is None or column not in row:
        return np.nan
    return row[column]

def layer_or_blank(row):
    if row is None:
        return pd.NA
    return row.get('layer_label', pd.NA)

def table_1a(best, probe=HEADLINE_PROBE):
    rows = []
    for model_key in MODEL_ORDER:
        meta = MODEL_META[model_key]
        mvp = row_for(best, 'mvp', model_key, probe)
        intphys = row_for(best, 'intphys2', model_key, probe)
        rows.append({
            'Model family': meta['label'],
            'Backbone': meta['paper_backbone'],
            'Pretraining objective': meta['objective'],
            'Best MVP layer': layer_or_blank(mvp),
            'MVP pair consistency': value_or_nan(mvp, 'primary_score'),
            'MVP accuracy': value_or_nan(mvp, 'test_accuracy_score'),
            'Best IntPhys2 layer': layer_or_blank(intphys),
            'IntPhys2 VOE': value_or_nan(intphys, 'primary_score'),
            'IntPhys2 accuracy': value_or_nan(intphys, 'test_accuracy_score'),
        })
    return pd.DataFrame(rows)

def table_1b(best, probe=HEADLINE_PROBE):
    rows = []
    for model_key in MODEL_ORDER:
        mvp = row_for(best, 'mvp', model_key, probe)
        intphys = row_for(best, 'intphys2', model_key, probe)
        rows.append({
            'model_key': model_key,
            'Model': MODEL_META[model_key]['label'],
            'MVP primary metric': value_or_nan(mvp, 'primary_score'),
            'IntPhys2 primary metric': value_or_nan(intphys, 'primary_score'),
        })
    out = pd.DataFrame(rows)
    out['MVP rank'] = out['MVP primary metric'].rank(ascending=False, method='min')
    out['IntPhys2 rank'] = out['IntPhys2 primary metric'].rank(ascending=False, method='min')
    return out.drop(columns='model_key')

def table_2a(best, probe=HEADLINE_PROBE):
    rows = []
    sub = best[best['probe'].eq(probe)]
    for model_key in MODEL_ORDER:
        mvp = row_for(sub, 'mvp', model_key)
        intphys = row_for(sub, 'intphys2', model_key)
        rows.append({
            'Model': MODEL_META[model_key]['label'],
            'Best layer on MVP': layer_or_blank(mvp),
            'Best MVP score': value_or_nan(mvp, 'primary_score'),
            'Best layer on IntPhys2': layer_or_blank(intphys),
            'Best IntPhys2 score': value_or_nan(intphys, 'primary_score'),
        })
    return pd.DataFrame(rows)

def classify_trend(values, tolerance=2.0):
    y = pd.Series(values).dropna().to_numpy(dtype=float)
    if len(y) < 3:
        return 'insufficient'
    if np.nanmax(y) - np.nanmin(y) <= tolerance:
        return 'flat'
    peak_idx = int(np.nanargmax(y))
    if peak_idx == len(y) - 1 and np.all(np.diff(y) >= -tolerance):
        return 'late-rising'
    if 0 < peak_idx < len(y) - 1 and y[peak_idx] - y[-1] > tolerance:
        return 'intermediate peak'
    return 'non-monotonic'

def table_2b(results, probe=HEADLINE_PROBE):
    rows = []
    sub = results[(results['probe'].eq(probe)) & (~results['is_ltx'])].copy()
    for model_key in [key for key in MODEL_ORDER if key != 'ltx_video']:
        row = {'Model': MODEL_META[model_key]['label']}
        labels = []
        for dataset in DATASET_ORDER:
            series = sub[(sub['dataset'].eq(dataset)) & (sub['model_key'].eq(model_key))].sort_values('relative_depth')['plot_test_primary']
            trend = classify_trend(series)
            row[f'{DATASET_LABEL[dataset]} depth trend'] = trend
            labels.append(trend)
        row['Peak location type'] = ' / '.join(labels)
        rows.append(row)
    return pd.DataFrame(rows)

def table_3a(best):
    rows = []
    for model_key in MODEL_ORDER:
        row = {'Model': MODEL_META[model_key]['label']}
        for dataset in DATASET_ORDER:
            scores = {}
            for probe in PROBE_ORDER:
                score = value_or_nan(row_for(best, dataset, model_key, probe), 'primary_score')
                scores[probe] = score
                row[f'{DATASET_LABEL[dataset]} {PROBE_LABEL[probe]} score'] = score
            valid = {probe: score for probe, score in scores.items() if pd.notna(score)}
            row[f'Winning probe on {DATASET_LABEL[dataset]}'] = PROBE_LABEL[max(valid, key=valid.get)] if valid else pd.NA
        rows.append(row)
    return pd.DataFrame(rows)

def table_3b(best):
    rows = []
    for model_key in MODEL_ORDER:
        row = {'Model': MODEL_META[model_key]['label']}
        for dataset in DATASET_ORDER:
            linear = value_or_nan(row_for(best, dataset, model_key, 'linear'), 'primary_score')
            mlp = value_or_nan(row_for(best, dataset, model_key, 'mlp'), 'primary_score')
            attn = value_or_nan(row_for(best, dataset, model_key, 'temporal_attn'), 'primary_score')
            row[f'{DATASET_LABEL[dataset]} MLP - linear'] = mlp - linear if pd.notna(mlp) and pd.notna(linear) else np.nan
            row[f'{DATASET_LABEL[dataset]} attentive - linear'] = attn - linear if pd.notna(attn) and pd.notna(linear) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

def annotation_color(value, norm, cmap_obj):
    rgba = cmap_obj(norm(value))
    luminance = 0.2126 * rgba[0] + 0.7152 * rgba[1] + 0.0722 * rgba[2]
    return 'black' if luminance > 0.58 else 'white'

def draw_heatmap(pivot, title, ax, cmap='viridis'):
    if pivot.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
        ax.set_title(title)
        ax.set_axis_off()
        return
    values = pivot.to_numpy(dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        ax.text(0.5, 0.5, 'No finite data', ha='center', va='center')
        ax.set_title(title)
        ax.set_axis_off()
        return
    vmin, vmax = float(np.nanmin(finite)), float(np.nanmax(finite))
    cmap_obj = plt.get_cmap(cmap)
    norm = plt.Normalize(vmin=vmin, vmax=vmax if vmax > vmin else vmin + 1.0)
    if sns is not None:
        sns.heatmap(pivot, annot=False, cmap=cmap, ax=ax, cbar=True, vmin=vmin, vmax=vmax)
    else:
        im = ax.imshow(values, aspect='auto', cmap=cmap_obj, norm=norm)
        ax.figure.colorbar(im, ax=ax)
        ax.set_xticks(np.arange(len(pivot.columns)), labels=pivot.columns, rotation=45, ha='right')
        ax.set_yticks(np.arange(len(pivot.index)), labels=pivot.index)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            value = values[i, j]
            if np.isfinite(value):
                ax.text(
                    j + 0.5 if sns is not None else j,
                    i + 0.5 if sns is not None else i,
                    f'{value:.1f}',
                    ha='center',
                    va='center',
                    color=annotation_color(value, norm, cmap_obj),
                    fontsize=9.0,
                )
    ax.set_title(title)

def plot_headline_dot(best, probe=HEADLINE_PROBE):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    model_labels = [MODEL_META[key]['label'] for key in MODEL_ORDER]
    y = np.arange(len(MODEL_ORDER))
    for ax, dataset in zip(axes, PANEL_DATASET_ORDER):
        scores = [value_or_nan(row_for(best, dataset, key, probe), 'primary_score') for key in MODEL_ORDER]
        stds = [value_or_nan(row_for(best, dataset, key, probe), 'primary_score_std') for key in MODEL_ORDER]
        colors = [MODEL_COLORS.get(key, '#777777') for key in MODEL_ORDER]
        for score, err, yi, color in zip(scores, stds, y, colors):
            if pd.notna(score):
                ax.errorbar(
                    score, yi,
                    xerr=None if pd.isna(err) else err,
                    fmt='o', markersize=8, color=color,
                    markeredgecolor='black', markeredgewidth=0.45,
                    ecolor=color, elinewidth=1.25, capsize=2.5, zorder=3,
                )
                ax.text(score + 0.8, yi, f'{score:.1f}', va='center', fontsize=9)
        ax.set_yticks(y, labels=model_labels)
        ax.invert_yaxis()
        ax.set_xlabel(PRIMARY_LABEL[dataset])
        ax.set_title(DATASET_LABEL[dataset])
    fig.suptitle(f'Experiment 1: headline comparison ({PROBE_LABEL[probe]})')
    fig.tight_layout()
    return fig

def depth_legend_handles(legend_handles, baseline_color):
    handles = [legend_handles[key] for key in MODEL_ORDER if key in legend_handles]
    labels = [MODEL_META[key]['label'] for key in MODEL_ORDER if key in legend_handles]
    handles.append(Line2D([0], [0], color=baseline_color, linestyle=(0, (4, 2)), linewidth=1.7))
    labels.append('Random baseline')
    return handles, labels

def draw_depth_panel(ax, data, family, dataset, legend_handles, *, compact=False, font_scale=1.0):
    ds = data[(data['family'].eq(family)) & (data['dataset'].eq(dataset))]
    baseline_color = REFERENCE_COLORS['random_baseline']
    for model_key in MODEL_ORDER:
        model_df = ds[ds['model_key'].eq(model_key)].sort_values('relative_depth')
        if model_df.empty:
            continue
        color = MODEL_COLORS.get(model_key, FAMILY_COLORS.get(family, '#777777'))
        line, = ax.plot(
            model_df['relative_depth'],
            model_df['plot_test_primary'],
            marker='o',
            markersize=(4.6 if compact else 5.2) * font_scale,
            linestyle='-',
            color=color,
            linewidth=(2.05 if compact else 2.25) * font_scale,
            label=MODEL_META[model_key]['label'],
            zorder=3,
        )
        legend_handles.setdefault(model_key, line)
    ax.axhline(
        RANDOM_BASELINE[dataset],
        color=baseline_color,
        linestyle=(0, (4, 2)),
        linewidth=1.7 * font_scale,
        alpha=0.98,
        zorder=5,
        dash_capstyle='butt',
    )
    ax.set_title(f'{family}\n{DATASET_LABEL[dataset]}', fontsize=(13.0 if compact else 13.5) * font_scale, fontweight='bold', pad=3)
    ax.set_xlabel('Relative depth', fontsize=(12.0 if compact else 12.5) * font_scale, labelpad=2)
    ax.set_ylabel(PRIMARY_LABEL[dataset], fontsize=(12.0 if compact else 12.5) * font_scale, labelpad=2)
    ax.set_xticks([0.25, 0.50, 0.75, 1.00])
    ax.tick_params(axis='both', labelsize=(11.0 if compact else 11.5) * font_scale, pad=2)
    ax.grid(axis='y', alpha=0.24, zorder=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

def plot_depth_by_family(results, probe=HEADLINE_PROBE):
    """Improved one-row, four-column version for direct paper-width comparison."""
    sub = results[(results['probe'].eq(probe)) & (~results['is_ltx'])].dropna(subset=['relative_depth', 'plot_test_primary']).copy()
    panels = [(family, dataset) for family in ['V-JEPA', 'VideoMAE'] for dataset in PANEL_DATASET_ORDER]
    fig, axes = plt.subplots(1, len(panels), figsize=(14.6, 4.05), squeeze=False, sharex=True)
    legend_handles = {}
    for ax, (family, dataset) in zip(axes.ravel(), panels):
        draw_depth_panel(ax, sub, family, dataset, legend_handles, compact=True, font_scale=1.18)
    handles, labels = depth_legend_handles(legend_handles, REFERENCE_COLORS['random_baseline'])
    fig.text(0.01, 0.985, f'{PROBE_LABEL[probe]} probe', ha='left', va='top', fontsize=16.0, fontweight='bold')
    fig.legend(
        handles,
        labels,
        loc='lower center',
        bbox_to_anchor=(0.5, 0.035),
        ncol=min(6, len(labels)),
        frameon=False,
        fontsize=14.0,
        columnspacing=1.15,
        handlelength=1.7,
        markerscale=1.15,
    )
    fig.tight_layout(rect=[0, 0.20, 1, 0.92], w_pad=1.0)
    return fig

def plot_depth_by_family_2x2(results, probe=HEADLINE_PROBE):
    """Two-by-two version with larger panel area and paper-readable fonts."""
    sub = results[(results['probe'].eq(probe)) & (~results['is_ltx'])].dropna(subset=['relative_depth', 'plot_test_primary']).copy()
    panels = [(family, dataset) for family in ['V-JEPA', 'VideoMAE'] for dataset in PANEL_DATASET_ORDER]
    fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.8), squeeze=False, sharex=True)
    legend_handles = {}
    for ax, (family, dataset) in zip(axes.ravel(), panels):
        draw_depth_panel(ax, sub, family, dataset, legend_handles, compact=False)
    handles, labels = depth_legend_handles(legend_handles, REFERENCE_COLORS['random_baseline'])
    fig.text(0.015, 0.99, f'{PROBE_LABEL[probe]} probe', ha='left', va='top', fontsize=13.0, fontweight='bold')
    fig.legend(
        handles,
        labels,
        loc='lower center',
        bbox_to_anchor=(0.5, 0.01),
        ncol=3,
        frameon=False,
        fontsize=11.3,
        columnspacing=1.0,
        handlelength=1.55,
    )
    fig.tight_layout(rect=[0, 0.16, 1, 0.94], w_pad=1.0, h_pad=1.0)
    return fig

def plot_depth_all_probes_appendix(results):
    """Appendix grid with probe columns and clean family/dataset row labels."""
    sub = results[(~results['is_ltx'])].dropna(subset=['relative_depth', 'plot_test_primary']).copy()
    row_panels = [(family, dataset) for family in ['V-JEPA', 'VideoMAE'] for dataset in PANEL_DATASET_ORDER]
    fig, axes = plt.subplots(len(row_panels), len(PROBE_ORDER), figsize=(11.8, 8.7), squeeze=False, sharex=True)
    legend_handles = {}
    for row_idx, (family, dataset) in enumerate(row_panels):
        for col_idx, probe in enumerate(PROBE_ORDER):
            ax = axes[row_idx, col_idx]
            panel_data = sub[sub['probe'].eq(probe)]
            draw_depth_panel(ax, panel_data, family, dataset, legend_handles, compact=True)
            ax.set_title('')
            ax.set_ylabel('')
            if row_idx < len(row_panels) - 1:
                ax.set_xlabel('')
            else:
                ax.set_xlabel('Relative depth', fontsize=11.6, labelpad=2)
    fig.tight_layout(rect=[0.145, 0.105, 0.995, 0.925], w_pad=0.85, h_pad=0.8)

    for col_idx, probe in enumerate(PROBE_ORDER):
        bbox = axes[0, col_idx].get_position()
        fig.text(
            (bbox.x0 + bbox.x1) / 2,
            0.965,
            PROBE_LABEL[probe],
            ha='center',
            va='top',
            fontsize=15.5,
            fontweight='bold',
        )
    for row_idx, (family, dataset) in enumerate(row_panels):
        bbox = axes[row_idx, 0].get_position()
        row_center = (bbox.y0 + bbox.y1) / 2
        metric = 'VOE accuracy' if dataset == 'intphys2' else 'pair consistency'
        fig.text(
            0.065,
            row_center + 0.018,
            f'{family}\n{DATASET_LABEL[dataset]}',
            ha='center',
            va='center',
            fontsize=12.2,
            fontweight='bold',
            linespacing=1.08,
        )
        fig.text(
            0.065,
            row_center - 0.043,
            f'({metric})',
            ha='center',
            va='center',
            fontsize=9.8,
            fontweight='normal',
            color='#5F6368',
        )

    handles, labels = depth_legend_handles(legend_handles, REFERENCE_COLORS['random_baseline'])
    fig.legend(
        handles,
        labels,
        loc='lower center',
        bbox_to_anchor=(0.5, 0.018),
        ncol=min(6, len(labels)),
        frameon=False,
        fontsize=11.4,
        columnspacing=1.05,
        handlelength=1.5,
    )
    return fig

def plot_depth_family_mean(results, probe=HEADLINE_PROBE):
    sub = results[(results['probe'].eq(probe)) & (~results['is_ltx'])].dropna(subset=['relative_depth', 'plot_test_primary']).copy()
    mean_df = (
        sub.groupby(['dataset', 'family', 'relative_depth'], as_index=False)
        .agg(plot_test_primary=('plot_test_primary', 'mean'))
    )
    fig, axes = plt.subplots(1, len(PANEL_DATASET_ORDER), figsize=(6.6, 2.85), squeeze=False, sharex=True)
    baseline_color = REFERENCE_COLORS['random_baseline']
    legend_handles = {}
    for ax, dataset in zip(axes.ravel(), PANEL_DATASET_ORDER):
        ds = mean_df[mean_df['dataset'].eq(dataset)]
        for family in ['V-JEPA', 'VideoMAE']:
            fam_df = ds[ds['family'].eq(family)].sort_values('relative_depth')
            if fam_df.empty:
                continue
            line, = ax.plot(
                fam_df['relative_depth'],
                fam_df['plot_test_primary'],
                marker='o',
                markersize=4.4,
                linestyle='-',
                color=FAMILY_COLORS[family],
                linewidth=2.1,
                label=f'{family} mean',
                zorder=3,
            )
            legend_handles.setdefault(family, line)
        ax.axhline(
            RANDOM_BASELINE[dataset],
            color=baseline_color,
            linestyle=(0, (4, 2)),
            linewidth=1.55,
            alpha=0.98,
            zorder=5,
            dash_capstyle='butt',
        )
        ax.set_title(DATASET_LABEL[dataset], fontsize=11.3, fontweight='bold', pad=2)
        ax.set_xlabel('Relative depth', fontsize=10.4, labelpad=2)
        ax.set_ylabel(PRIMARY_LABEL[dataset], fontsize=10.4, labelpad=2)
        ax.set_xticks([0.25, 0.50, 0.75, 1.00])
        ax.tick_params(axis='both', labelsize=9.4, pad=2)
        ax.grid(axis='y', alpha=0.24, zorder=0)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    handles = [legend_handles[family] for family in ['V-JEPA', 'VideoMAE'] if family in legend_handles]
    labels = [f'{family} mean' for family in ['V-JEPA', 'VideoMAE'] if family in legend_handles]
    handles.append(Line2D([0], [0], color=baseline_color, linestyle=(0, (4, 2)), linewidth=1.55))
    labels.append('Random baseline')
    fig.text(0.01, 0.985, f'{PROBE_LABEL[probe]} probe', ha='left', va='top', fontsize=11.2, fontweight='bold')
    fig.legend(
        handles,
        labels,
        loc='lower center',
        bbox_to_anchor=(0.5, 0.015),
        ncol=len(labels),
        frameon=False,
        fontsize=9.8,
        columnspacing=1.2,
        handlelength=1.5,
    )
    fig.tight_layout(rect=[0, 0.15, 1, 0.94], w_pad=1.15)
    return fig


def plot_depth_heatmap(results, probe=HEADLINE_PROBE):
    sub = results[(results['probe'].eq(probe)) & (~results['is_ltx'])].dropna(subset=['relative_depth_label', 'plot_test_primary']).copy()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, dataset in zip(axes, PANEL_DATASET_ORDER):
        ds = sub[sub['dataset'].eq(dataset)].copy()
        ds['Model'] = ds['model_key'].map(lambda key: MODEL_META[key]['label'])
        depth_cols = sorted(ds['relative_depth_label'].dropna().unique(), key=lambda value: float(value))
        model_rows = [MODEL_META[key]['label'] for key in MODEL_ORDER if key != 'ltx_video']
        pivot = ds.pivot_table(index='Model', columns='relative_depth_label', values='plot_test_primary', aggfunc='max').reindex(index=model_rows, columns=depth_cols)
        draw_heatmap(pivot, f'{DATASET_LABEL[dataset]} ({PRIMARY_LABEL[dataset]})', ax)
        ax.set_xlabel('Relative depth slot')
        ax.set_ylabel('Model')
    fig.suptitle(f'Experiment 2: model x depth heatmap ({PROBE_LABEL[probe]})')
    fig.tight_layout()
    return fig

def plot_ltx_heatmap(results, probe=HEADLINE_PROBE):
    sub = results[(results['probe'].eq(probe)) & (results['model_key'].eq('ltx_video'))].dropna(subset=['ltx_noise_level', 'ltx_block', 'plot_test_primary']).copy()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, dataset in zip(axes, PANEL_DATASET_ORDER):
        ds = sub[sub['dataset'].eq(dataset)].copy()
        noise = sorted(ds['ltx_noise_level'].dropna().unique(), reverse=True)
        blocks = sorted(ds['ltx_block'].dropna().unique())
        pivot = ds.pivot_table(index='ltx_noise_level', columns='ltx_block', values='plot_test_primary', aggfunc='max').reindex(index=noise, columns=blocks)
        draw_heatmap(pivot, f'{DATASET_LABEL[dataset]} ({PRIMARY_LABEL[dataset]})', ax, cmap='magma')
        ax.set_xlabel('Transformer block')
        ax.set_ylabel('Noise level')
    fig.tight_layout()
    return fig

def plot_probe_performance_bars(best):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
    for ax, dataset in zip(axes, PANEL_DATASET_ORDER):
        ds = best[best['dataset'].eq(dataset)].copy()
        ds['Model'] = ds['model_key'].map(lambda key: MODEL_META[key]['label'])
        ds['Probe'] = ds['probe'].map(PROBE_LABEL)
        model_rows = [MODEL_META[key]['label'] for key in MODEL_ORDER]
        probe_cols = [PROBE_LABEL[key] for key in PROBE_ORDER]
        plot_df = ds.pivot_table(index='Model', columns='Probe', values='primary_score', aggfunc='max').reindex(index=model_rows, columns=probe_cols)
        err_df = ds.pivot_table(index='Model', columns='Probe', values='primary_score_std', aggfunc='max').reindex(index=model_rows, columns=probe_cols)
        plot_df.plot(
            kind='bar',
            ax=ax,
            color=[PROBE_COLORS[p] for p in PROBE_ORDER],
            edgecolor='black',
            linewidth=0.35,
            yerr=None,
        )
        n_models = len(model_rows)
        n_probes = len(probe_cols)
        bar_width = 0.8 / n_probes
        for probe_idx, probe_label in enumerate(probe_cols):
            x_positions = np.arange(n_models) - 0.4 + bar_width * (probe_idx + 0.5)
            for model_idx, model_label in enumerate(model_rows):
                score = plot_df.loc[model_label, probe_label]
                err = err_df.loc[model_label, probe_label]
                if pd.notna(score) and pd.notna(err) and float(err) > 0:
                    ax.errorbar(
                        x_positions[model_idx],
                        float(score),
                        yerr=float(err),
                        fmt='none',
                        ecolor='#111111',
                        elinewidth=1.35,
                        capsize=4.0,
                        capthick=1.35,
                        zorder=8,
                    )
        ax.set_title(f'{DATASET_LABEL[dataset]} ({PRIMARY_LABEL[dataset]})')
        ax.set_ylabel('Primary metric')
        ax.set_xlabel('Model')
        ax.tick_params(axis='x', rotation=45)
        ax.legend(title='Probe', loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0)
    fig.suptitle('Experiment 3: probe performance by model')
    fig.tight_layout(rect=[0, 0, 0.88, 1])
    return fig

def plot_probe_heatmap(best):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, dataset in zip(axes, PANEL_DATASET_ORDER):
        sub = best[best['dataset'].eq(dataset)].copy()
        sub['Model'] = sub['model_key'].map(lambda key: MODEL_META[key]['label'])
        sub['Probe'] = sub['probe'].map(PROBE_LABEL)
        model_rows = [MODEL_META[key]['label'] for key in MODEL_ORDER]
        probe_cols = [PROBE_LABEL[key] for key in PROBE_ORDER]
        pivot = sub.pivot_table(index='Model', columns='Probe', values='primary_score', aggfunc='max').reindex(index=model_rows, columns=probe_cols)
        draw_heatmap(pivot, f'{DATASET_LABEL[dataset]} ({PRIMARY_LABEL[dataset]})', ax, cmap='crest' if sns is not None else 'viridis')
        ax.set_xlabel('Probe')
        ax.set_ylabel('Model')
    fig.suptitle('Experiment 3: probe ranking heatmap')
    fig.tight_layout()
    return fig

all_results = load_excel_results()
results = paper_results(all_results)
main_seed_coverage_issues = report_seed_coverage(
    results.dropna(subset=['plot_test_primary']),
    'main.ipynb paper-backbone layerwise/depth plot rows',
    key_columns=['dataset', 'experiment', 'model_label', 'backbone', 'probe', 'layer', 'layer_label', 'layer_key'],
)

for probe in PROBE_ORDER:
    plot_depth_by_family(results, probe)
    plt.show()
    plot_depth_by_family_2x2(results, probe)
    plt.show()
    plot_depth_family_mean(results, probe)
    plt.show()
    plot_ltx_heatmap(results, probe)
    plt.show()

plot_depth_all_probes_appendix(results)
plt.show()


In [ ]:
# Table 1 display + LaTeX copy cell
# Flags for paper-table variants.
TABLE1_INCLUDE_VARIANCE = True
TABLE1_USE_ATTENTIVE_SEEDED_MEAN = True


def apply_table1_attentive_policy(df, *, use_seeded_mean=True):
    out = df.copy()
    if use_seeded_mean:
        return out
    mask = out['probe'].eq('temporal_attn')
    out.loc[mask, 'plot_test_primary'] = out.loc[mask, 'test_primary']
    out.loc[mask, 'plot_test_primary_std'] = np.nan
    out.loc[mask, 'plot_test_accuracy'] = out.loc[mask, 'test_accuracy']
    out.loc[mask, 'plot_test_accuracy_std'] = np.nan
    out.loc[mask, 'plot_val_primary'] = out.loc[mask, 'val_primary']
    out.loc[mask, 'plot_val_primary_std'] = np.nan
    out.loc[mask, 'seed_source'] = 'original_seed42'
    return out


_table1_all_results = apply_table1_attentive_policy(
    attach_seed_stats(load_excel_results()),
    use_seeded_mean=TABLE1_USE_ATTENTIVE_SEEDED_MEAN,
)
_table1_results = paper_results(_table1_all_results)
_table1_best = select_best_layers(_table1_results)
if TABLE1_INCLUDE_VARIANCE:
    _table1_seed_coverage_issues = report_seed_coverage(
        _table1_best,
        'main.ipynb Table 1 selected rows',
        key_columns=['dataset', 'experiment', 'model_label', 'backbone', 'probe', 'layer', 'layer_label', 'layer_key'],
    )
else:
    _table1_seed_coverage_issues = pd.DataFrame()


def _fmt_table1_layer(row):
    value = layer_or_blank(row)
    if pd.isna(value):
        return 'NA'
    text = str(value)
    numeric = pd.to_numeric(pd.Series([text]), errors='coerce').iloc[0]
    if pd.notna(numeric):
        return f'L{numeric:g}'
    return text.replace('Final layer', 'final').replace('Layer ', 'L')


def _fmt_table1_score(row, mean_col='primary_score', std_col='primary_score_std', latex=False, include_variance=True):
    value = value_or_nan(row, mean_col)
    if pd.isna(value):
        return 'NA'
    std = value_or_nan(row, std_col) if include_variance else np.nan
    if pd.notna(std):
        pm = r' \pm ' if latex else ' +/- '
        return f'{float(value):.2f}{pm}{float(std):.2f}'
    return f'{float(value):.2f}'


def _fmt_table1_cell(row, latex=False, include_variance=True):
    if row is None:
        return 'NA'
    score = _fmt_table1_score(row, latex=latex, include_variance=include_variance)
    layer = _fmt_table1_layer(row)
    if score == 'NA':
        return 'NA'
    if layer == 'NA':
        return score
    return f'{score} ({layer})'


def build_paper_table1(best, latex=False, include_variance=True):
    rows = []
    for model_key in MODEL_ORDER:
        meta = MODEL_META[model_key]
        row = {
            ('Model', ''): meta['label'],
            ('Backbone', ''): meta['paper_backbone'],
        }
        for probe in PROBE_ORDER:
            probe_label = PROBE_LABEL[probe]
            mvp = row_for(best, 'mvp', model_key, probe)
            intphys = row_for(best, 'intphys2', model_key, probe)
            row[(probe_label, 'MVP pair')] = _fmt_table1_cell(mvp, latex=latex, include_variance=include_variance)
            row[(probe_label, 'IntPhys2 VOE')] = _fmt_table1_cell(intphys, latex=latex, include_variance=include_variance)
        rows.append(row)
    out = pd.DataFrame(rows)
    out.columns = pd.MultiIndex.from_tuples(out.columns)
    return out


table1_display = build_paper_table1(_table1_best, latex=False, include_variance=TABLE1_INCLUDE_VARIANCE)
table1_latex_df = build_paper_table1(_table1_best, latex=True, include_variance=TABLE1_INCLUDE_VARIANCE)
table1_latex = table1_latex_df.to_latex(
    index=False,
    escape=False,
    multicolumn=True,
    multicolumn_format='c',
    column_format='ll' + 'cc' * len(PROBE_ORDER),
)

_policy = 'seeded mean/std for attentive' if TABLE1_USE_ATTENTIVE_SEEDED_MEAN else 'original seed-42 attentive'
_var = 'with variance' if TABLE1_INCLUDE_VARIANCE else 'without variance'
display(Markdown(f'**Table 1 draft** | layout: probe blocks | {_policy} | {_var}'))
display(table1_display)
print(table1_latex)
